# ChakraModel YOLO Cross-Dataset Training (Optimized for Attached Dataset)

This notebook trains a YOLOv8 object detector using the `chakramodel-evaluation-datasets` you directly attached to Kaggle.

**Instructions:**
1. Ensure **GPU T4 x2** (or P100) is selected as the accelerator.
2. Run all cells below.

In [1]:
!pip install ultralytics opencv-python-headless -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.8 MB/s eta 0:00:00


In [2]:
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO

out_dataset_dir = Path("/kaggle/working/polyp_dataset")
out_images_dir = out_dataset_dir / "images"
out_labels_dir = out_dataset_dir / "labels"

# Create YOLO directory structure
for split in ['train', 'val']:
    (out_images_dir / split).mkdir(parents=True, exist_ok=True)
    (out_labels_dir / split).mkdir(parents=True, exist_ok=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


### Step 1: Discover Attached Datasets

In [3]:
# Try to find the dataset root dynamically
possible_paths = [
    Path("/kaggle/input/chakramodel-evaluation-datasets"),
    Path("/kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets")
]

dataset_root = None
for p in possible_paths:
    if p.exists() and list(p.glob("*")):
        dataset_root = p
        break

if dataset_root is None:
    # Fallback search if path is slightly different
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'kvasir-seg' in dirs or 'cvc-clinicdb' in dirs:
            dataset_root = Path(root)
            break

print(f"Using dataset root: {dataset_root}")

Using dataset root: /kaggle/input/datasets/gokulrocky/chakramodel-evaluation-datasets


### Step 2: Convert Masks to YOLO Bounding Boxes

In [4]:
def mask_to_yolo(mask_path, img_w, img_h):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 0 and h > 0:
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            bboxes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    return bboxes

all_data = []

for dataset_folder in dataset_root.iterdir():
    if not dataset_folder.is_dir(): continue
    images_dir = dataset_folder / "images"
    masks_dir = dataset_folder / "masks"
    
    if images_dir.exists() and masks_dir.exists():
        for img_p in images_dir.glob("*"):
            if img_p.suffix.lower() in ['.jpg', '.png', '.jpeg']:
                # Handle case where mask has different extension than image (e.g. img.jpg vs mask.png)
                mask_matches = list(masks_dir.glob(f"{img_p.stem}.*"))
                if mask_matches:
                    all_data.append((img_p, mask_matches[0]))

print(f"Found {len(all_data)} images with masks across all datasets.")

Found 1500 images with masks across all datasets.


In [5]:
np.random.seed(42)
np.random.shuffle(all_data)

train_split = int(0.85 * len(all_data))

for i, (img_p, msk_p) in enumerate(all_data):
    split = "train" if i < train_split else "val"
    
    img = cv2.imread(str(img_p))
    if img is None: continue
    h, w = img.shape[:2]
    
    yolo_lines = mask_to_yolo(msk_p, w, h)
    if not yolo_lines: continue
    
    new_img_name = f"{img_p.stem}_{i}.jpg"
    new_txt_name = f"{img_p.stem}_{i}.txt"
    
    # Copy image to YOLO directory
    shutil.copy2(img_p, out_images_dir / split / new_img_name)
    
    # Write YOLO label
    with open(out_labels_dir / split / new_txt_name, "w") as f:
        f.write("\n".join(yolo_lines))

print("YOLO Dataset generated successfully in /kaggle/working/polyp_dataset!")

YOLO Dataset generated successfully in /kaggle/working/polyp_dataset!


In [6]:
yaml_content = f"""path: {out_dataset_dir}
train: images/train
val: images/val

names:
  0: polyp
"""
with open(out_dataset_dir / "dataset.yaml", "w") as f:
    f.write(yaml_content)

### Step 3: Train YOLOv8 Model (Multi-GPU)

In [7]:
model = YOLO("yolov8n.pt") # Using nano for speed

# Uses device=[0, 1] to train on 2 GPUs in parallel
results = model.train(
    data=str(out_dataset_dir / "dataset.yaml"),
    epochs=50,
    imgsz=640,
    batch=32, # Batch size for dual GPU
    device=[0, 1], # Utilizes both GPUs
    project="/kaggle/working/yolo_output",
    name="cross_dataset_polyp"
)

print("Training Complete! Download the weights from /kaggle/working/yolo_output/cross_dataset_polyp/weights/best.pt")

Ultralytics 8.4.140 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/polyp_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, m

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      3.36G      1.672      1.935      1.354        387        640: 100% ━━━━━━━━━━━━ 40/40 4.5it/s 8.9s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.6it/s 1.1s0.4s
                   all        225      12131      0.127     0.0161    0.00954     0.0061

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      3.36G      1.814      1.857      1.342        249        640: 100% ━━━━━━━━━━━━ 40/40 4.4it/s 9.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.6it/s 1.1s0.4s
                   all        225      12131      0.118    0.00981    0.00615    0.00426

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      3.36G      1.923      1.847      1.351        296        640: 100% ━━━